In [1]:
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
import pandas as pd
import pickle
import numpy as np
import os
from pathlib import Path

result_dir = Path('result/custom')
results = []

for dma_dir in result_dir.iterdir():
    if not dma_dir.is_dir():
        continue
    
    dma = dma_dir.name


    # Check if files exist
    if not (dma_dir / 'f_beta.pkl').exists():
        continue

    with open(dma_dir / 'f_beta.pkl', 'rb') as f:
        f_beta = pickle.load(f)
    with open(dma_dir / 'precision.pkl', 'rb') as f:
        precision = pickle.load(f)
    with open(dma_dir / 'recall.pkl', 'rb') as f:
        recall = pickle.load(f)
   
    with open(dma_dir / 'optimal_threshold.pkl', 'rb') as f:
        optimal_threshold = pickle.load(f)

    # 1. Load scores to find the max value (needed to reconstruct thresholds)
    with open(dma_dir / 'score.pkl', 'rb') as f:
        scores = pickle.load(f)

    with open(dma_dir / 'thresholds_list.pkl', 'rb') as f:
        thresholds_list = pickle.load(f)
    
    # robustly find max score whether it's a list of tensors or numpy arrays
    flat_scores = []
    for s in scores:
        if hasattr(s, 'cpu'): s = s.cpu().numpy() # handle torch tensor
        flat_scores.extend(np.array(s).flatten())
    max_score = np.max(flat_scores) if len(flat_scores) > 0 else 1.0


    if len(f_beta) == 0:
        results.append({'dma': dma, 'f_beta': None, 'precision': None, 'recall': None, 'threshold': None})
        continue

    print("Threshold:", optimal_threshold)
    
    if isinstance(f_beta, list):
        f_beta = np.array(f_beta)
        f_beta = f_beta[0]
    if isinstance(precision, list):
        precision = np.array(precision)
        precision = precision[0]
    if isinstance(recall, list):
        recall = np.array(recall)
        recall = recall[0]
    
    print('fbeta shape:', f_beta.shape)

    # 2. Reconstruct thresholds array and find index
    # (assumes linear spacing as used in get_precision_recall)
    thresholds = np.array(thresholds_list)
    # Find index where threshold is closest to optimal_threshold
    idx = (np.abs(thresholds - optimal_threshold)).argmin()

    best_f = f_beta[idx]
    best_prec = precision[idx]
    best_rec = recall[idx]
    
    results.append({
        'dma': dma, 
        'f_beta': best_f, 
        'precision': best_prec, 
        'recall': best_rec,
        'threshold': optimal_threshold
    })
  

df = pd.DataFrame(results)
# Sort by dma name (handling numbers correctly if possible, but simple sort is fine for now)
df = df.sort_values('dma').reset_index(drop=True)
df

Threshold: 108.52217864990234
fbeta shape: (999,)
Threshold: 399.0337829589844
fbeta shape: (999,)
Threshold: 268.49188232421875
fbeta shape: (1000,)
Threshold: 257.3085632324219
fbeta shape: (1000,)
Threshold: 141.92623901367188
fbeta shape: (1000,)
Threshold: 131.82627868652344
fbeta shape: (983,)
Threshold: 173.67343139648438
fbeta shape: (999,)
Threshold: 175.47569274902344
fbeta shape: (1000,)
Threshold: 156.2273712158203
fbeta shape: (897,)
Threshold: 1.0
fbeta shape: (1000,)
Threshold: 1.0
fbeta shape: (976,)
Threshold: 399.0337829589844
fbeta shape: (999,)
Threshold: 1.0
fbeta shape: (1000,)
Threshold: 1.0373917818069458
fbeta shape: (999,)
Threshold: 1.0
fbeta shape: (999,)
Threshold: 30.689228057861328
fbeta shape: (842,)
Threshold: 1.0
fbeta shape: (1000,)
Threshold: 111.01838684082031
fbeta shape: (1000,)
Threshold: 131.32669067382812
fbeta shape: (965,)
Threshold: 201.77586364746094
fbeta shape: (999,)
Threshold: 205.2840576171875
fbeta shape: (999,)
Threshold: 204.0133666

,dma,f_beta,precision,recall,threshold
0,Area_1,0.652765,0.484522,1.000000,1.000000
1,Area_1_run1,0.652765,0.484522,1.000000,1.000000
2,Area_1_run2,0.655144,0.487148,1.000000,1.000000
3,Area_1_run3,0.652900,0.484671,1.000000,1.000000
4,Area_1_run4,0.654462,0.486395,1.000000,1.008751
5,Area_1_run5,0.655075,0.487072,1.000000,1.000000
6,Area_2,0.128225,0.473988,0.074141,126.435547
7,Area_2_run1,0.128225,0.473988,0.074141,126.435547
8,Area_2_run2,0.054561,0.477612,0.028933,257.308563
9,Area_2_run3,0.068027,0.571429,0.036166,268.491882


In [9]:
areas = range(1, 6 + 1)
runs = range(1, 5 + 1)
results = {}
for area in areas:
    f_score = []
    precision = []
    recall = []
    for run in runs:
        dma_name = f'Area_{area}_run{run}'
        dma_result = df[df['dma'] == dma_name]
        if not dma_result.empty:
            f_score.append(dma_result['f_beta'].values[0])
            precision.append(dma_result['precision'].values[0])
            recall.append(dma_result['recall'].values[0])
    results[f'Area_{area}'] = {
        'f_beta': f_score,
        'precision': precision,
        'recall': recall
    }
        


In [10]:
print("Dma | F-beta | Precision | Recall")
for area, metrics in results.items():
    # print(f"Results for {area}:")
    # print(f"  F-beta scores: {np.mean(metrics['f_beta']):.4f} ± {np.std(metrics['f_beta']):.4f}")
    # print(f"  Precision: {np.mean(metrics['precision']):.4f} ± {np.std(metrics['precision']):.4f}")
    # print(f"  Recall: {np.mean(metrics['recall']):.4f} ± {np.std(metrics['recall']):.4f}")
    print(f"{area} | {np.mean(metrics['f_beta']):.4f} ± {np.std(metrics['f_beta']):.4f} | {np.mean(metrics['precision']):.4f} ± {np.std(metrics['precision']):.4f} | {np.mean(metrics['recall']):.4f} ± {np.std(metrics['recall']):.4f}")

Dma | F-beta | Precision | Recall
Area_1 | 0.6541 ± 0.0010 | 0.4860 ± 0.0011 | 1.0000 ± 0.0000
Area_2 | 0.0938 ± 0.0282 | 0.4572 ± 0.0714 | 0.0533 ± 0.0179
Area_3 | 0.3939 ± 0.0851 | 0.7666 ± 0.0264 | 0.2698 ± 0.0802
Area_4 | 0.2915 ± 0.0034 | 0.1726 ± 0.0015 | 0.9368 ± 0.0259
Area_5 | 0.4520 ± 0.1476 | 0.3896 ± 0.1643 | 0.6524 ± 0.0867
Area_6 | 0.3061 ± 0.0184 | 0.3278 ± 0.0468 | 0.2907 ± 0.0105
